# ResNet 실습: 스킵 커넥션이 기울기를 살리는 이유

깊은 네트워크에서 residual 구조가 기울기 흐름을 어떻게 돕는지 숫자로 확인합니다.


In [ ]:
# numpy로 깊은 층의 기울기 곱을 시뮬레이션하고 matplotlib으로 시각화합니다.
import numpy as np
import matplotlib.pyplot as plt

# 재현 가능한 실험을 위해 시드를 고정합니다.
np.random.seed(21)


In [ ]:
# 일반 네트워크는 각 층의 기울기가 계속 곱해집니다.
depths = np.arange(1, 101)
trials = 2000
plain_gradients = []
residual_gradients = []

for depth in depths:
    # 작은 값들이 반복해서 곱해지는 plain chain을 만듭니다.
    local = np.random.normal(loc=0.9, scale=0.08, size=(trials, depth))
    plain_gradients.append(np.median(np.abs(np.prod(local, axis=1))))

    # residual chain은 항등 경로 때문에 1 + 작은 변화량처럼 볼 수 있습니다.
    residual_local = 1 + np.random.normal(loc=0.0, scale=0.08, size=(trials, depth))
    residual_gradients.append(np.median(np.abs(np.prod(residual_local, axis=1))))

plt.plot(depths, plain_gradients, label="plain chain")
plt.plot(depths, residual_gradients, label="residual-like chain")
plt.yscale("log")
plt.xlabel("depth")
plt.ylabel("median absolute gradient")
plt.title("gradient flow with depth")
plt.legend()
plt.show()


In [ ]:
# residual block의 핵심 계산 F(x) + x를 작은 벡터로 구현합니다.
def residual_block(x, weight, bias):
    # F(x)는 학습되는 변환입니다. 여기서는 단순한 선형 변환과 ReLU로 표현합니다.
    fx = np.maximum(x @ weight + bias, 0)
    # 스킵 커넥션은 입력 x를 출력에 그대로 더합니다.
    return fx + x

x = np.array([[1.0, -0.5, 0.2, 0.7]])
weight = np.random.normal(0, 0.1, (4, 4))
bias = np.zeros((1, 4))
plain_out = np.maximum(x @ weight + bias, 0)
residual_out = residual_block(x, weight, bias)
print("입력 x:", np.round(x, 3))
print("일반 블록 F(x):", np.round(plain_out, 3))
print("잔차 블록 F(x) + x:", np.round(residual_out, 3))


## 관찰 포인트

ResNet의 핵심은 복잡한 층 자체보다 입력과 기울기가 지나갈 길을 열어둔 구조적 재표현입니다.
